## Import

In [1]:
import os
import time
import numpy as np
import scipy.linalg as linalg
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
#import preprocess


device = 'cuda:1'

%load_ext autoreload
%autoreload 2

## Experiment settings

In [3]:
from datasets.NLP import load_imdb_bert_dataset 

Xs1, Ys1, Lx1, Ly1 = load_imdb_bert_dataset(percent_shuffle=0.1, data_range=[0, 2000])
Xs2, Ys2, Lx2, Ly2 = load_imdb_bert_dataset(percent_shuffle=0.1, data_range=[4000, 6000])
Xs3, Ys3, Lx3, Ly3 = load_imdb_bert_dataset(percent_shuffle=0.1, data_range=[8000, 10000])

X, Y = torch.cat([Xs1, Xs2, Xs3], dim=1), torch.cat([Ys1, Ys2, Ys3], dim=1)

print(X.size(), Y.size())

class 0 torch.Size([12500, 768]) [0, 0, 0, 0, 0]
class 1 torch.Size([12500, 768]) [1, 1, 1, 1, 1]
X (4000, 768) Y (4000, 768)
Sanity check (percent identical labels, should be ~0.9500000000000001): 0.9435
class 0 torch.Size([12500, 768]) [0, 0, 0, 0, 0]
class 1 torch.Size([12500, 768]) [1, 1, 1, 1, 1]
X (4000, 768) Y (4000, 768)
Sanity check (percent identical labels, should be ~0.9500000000000001): 0.949
class 0 torch.Size([12500, 768]) [0, 0, 0, 0, 0]
class 1 torch.Size([12500, 768]) [1, 1, 1, 1, 1]
X (4000, 768) Y (4000, 768)
Sanity check (percent identical labels, should be ~0.9500000000000001): 0.9465
torch.Size([4000, 2304]) torch.Size([4000, 2304])


## Dimensionality reduction

In [4]:
n, d = torch.cat([X, Y], dim=1).size()

In [5]:
from dr.Autoencoder import Autoencoder
from utils import utils_os


d_array = [4, 8, 16, 32, 64]
mse_array = {}


for latent in d_array:
    # [A]. Train autoencoder to compress data
    ae = Autoencoder(x_dim=d//2, y_dim=d//2, latent_size=latent, alpha=1, lam=0)

    ae.to(device)
    X, Y = X.to(device).float(), Y.to(device).float()
    ae.learn(X, Y)

    XX, YY = ae.encode(X, Y)
    XX, YY = XX.clone().detach(), YY.clone().detach()
    X_rec, Y_rec = ae.decode(XX, YY)
    
    # [B]. calculate reconstruction loss
    loss_x = ae.rec_loss(X_rec, X, normalize=True).item()
    loss_y = ae.rec_loss(Y_rec, Y, normalize=True).item()
    
    mse_array[latent] = (loss_x, loss_y)

    print('d=', latent, 'mse=', mse_array[latent])

finished: t= 0 loss= 1.2433750629425049 loss val= 1.2436094284057617 best val loss= 1.2436094284057617 best t= 0
finished: t= 101 loss= 0.0028254336211830378 loss val= 0.0037326025776565075 best val loss= 0.0037326025776565075 best t= 101
finished: t= 202 loss= 0.0021491642110049725 loss val= 0.003011994296684861 best val loss= 0.0027672061696648598 best t= 186
finished: t= 303 loss= 0.0017418339848518372 loss val= 0.002499646507203579 best val loss= 0.002422971185296774 best t= 257
finished: t= 404 loss= 0.00216037780046463 loss val= 0.0030075660906732082 best val loss= 0.0023304042406380177 best t= 370
finished: t= 505 loss= 0.0014722218038514256 loss val= 0.002490096725523472 best val loss= 0.002317330799996853 best t= 430
finished: t= 606 loss= 0.0016952581936493516 loss val= 0.0029470522422343493 best val loss= 0.0022730580531060696 best t= 508
finished: t= 707 loss= 0.001599862240254879 loss val= 0.002781355520710349 best val loss= 0.002184836659580469 best t= 701
finished: t= 80